Direct YML Files analysis or keywords detection
in this method we do yml's analysis for separate detection of device setup and test command


In [1]:
# -*- coding: utf-8 -*-
import os
import re
import pandas as pd
from typing import List, Pattern, Tuple, Dict, Any

# === CONFIGURATION ===
CONFIG_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(OUTPUT_DIR, "3.1.1_YML_Files_V1.0.csv")
MISSES_CSV = os.path.join(OUTPUT_DIR, "3.1_YML_Missed_Triggers_Audit.csv")
MISSES_SUMMARY_CSV = os.path.join(OUTPUT_DIR, "3.1_YML_Missed_Triggers_Summary.csv")

# Tiny polish applied: anywhere triggers moved to fallback; flags added
# Search_Method_Name = (
#     "Phase 1 (v2.1) - Direct YAML Scan "
#     "[normalize run/script + multi-module + GMD deviceCheck + generic *AndroidTest (excludes) "
#     "+ var-hinted gradle + stronger GHA inputs + Spoon/Marathon/Flank + audit] "
#     "(anywhere triggers only in fallback; flags: trigger_via_prefix/anywhere)"
# )

# --- helpers ---
def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: List[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: List[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

# === NORMALIZATION ===
def normalize_block_keys(text: str) -> str:
    # Single-line: "run: ./gradlew ..." -> "./gradlew ..."
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\|)\s*(.+)$', r'\2', text)
    # Multiline: drop the key line only (keep content)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*\|?\s*$', '', text)
    return text

# Gradle prefix: allow env=..., sudo, bash/sh -c, pre-commands with &&, cd &&, wrapper
GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)
GRADLE_ANYWHERE = r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*'

# === DEVICE SOURCES ===
DEVICE_SOURCES = [
    ("Real_Device", "adb devices",      [r'(?m)^\s*adb\s+devices\b']),
    ("Real_Device", "adb get-state",    [r'(?m)^\s*adb\s+get-state\b']),
    ("Real_Device", "adb get-serialno", [r'(?m)^\s*adb\s+get-serialno\b']),
    ("Real_Device", "adb -s <serial> (physical)", [r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b']),
    ("Real_Device", "adb install",      [r'(?m)^\s*adb\s+install(\s+-r)?\b']),
    ("Real_Device", "adb shell",        [r'(?m)^\s*adb\s+shell\b']),
    ("Real_Device", "adb root",         [r'(?m)^\s*adb\s+root\b']),
    ("Real_Device", "adb settings",     [r'(?m)^\s*adb\s+shell\s+settings\b']),
    ("Real_Device", "adb input",        [r'(?m)^\s*adb\s+shell\s+input\b']),
    ("Real_Device", "adb pm grant",     [r'(?m)^\s*adb\s+shell\s+pm\s+grant\b']),

    ("Emulator", "adb -s emulator-serial", [
        r'(?m)^\s*adb\s+-s\s+emulator-\d+\b',
        r'(?m)^\s*adb\s+-s\s+(?:localhost|127\.0\.0\.1):\d+\b',
    ]),
    ("Emulator", "emulator -avd/@", [r'(?m)^\s*\S*emulator\b[^\n]*\s(-avd|@)\S+']),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh", [r'(?m)^\s*start-emulator\.sh\b']),
    ("Emulator", "android create avd", [r'(?m)^\s*\S*android\b[^\n]*\bcreate\s+avd\b']),
    ("Emulator", "circle-android wait-for-boot",[r'(?m)^\s*circle-android\s+wait-for-boot\b']),
    ("Emulator", "reactivecircus runner", [r'uses:\s*reactivecircus/android-emulator-runner']),
    ("Emulator", "sys-img component", [
        r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
        r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r'^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
    ]),
    ("Emulator", "api-level", [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}']),
    ("Emulator", "abi/arch",  [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image", [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name", [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),
    ("GMD", "managedDevices DSL",       [r'\bmanageddevices?\b']),
    ("GMD", "ManagedVirtualDevice DSL", [r'\bmanagedvirtualdevice\b|\bcom\.android\.build\.api\.dsl\.ManagedVirtualDevice\b']),
    ("GMD", "GMD task mentions",        [r'\bmanageddevice\w*androidtest\b']),
    ("GMD", "GHA gradle arguments/tasks", [
        r'(?m)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'(?m)^\s*tasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b'
    ]),
    ("Third_Party_Lab", "gcloud firebase", [r'(?m)^\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",        [r'(?m)^\s*saucectl(\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack",[r'\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",  [r'(?m)^\s*appcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",   [r'(?m)^\s*maestro\s+cloud\b']),
    ("Third_Party_Lab", "test_matrix/firebase.json", [r'\btest_matrix\.json\b|\bfirebase\.json\b']),
]

# === TRIGGER SOURCES (PRIMARY / high-confidence) ===
NON_TEST_PREFIX = r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)'

TRIGGER_SOURCES_PRIMARY = [
    # Connected
    ("Gradle", "connectedAndroidTest",                 [rf'(?m){GRADLE_PREFIX}[^\n]*\bconnectedandroidtest\b']),
    ("Gradle", "connected.*Android.*",                 [rf'(?m){GRADLE_PREFIX}[^\n]*\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle", "connectedCheck",                       [rf'(?m){GRADLE_PREFIX}\s+(?::[\w-]+:)*connectedcheck\b']),
    ("Gradle", "connectedAndroidTest (abbr)",          [rf'(?m){GRADLE_PREFIX}[^\n]*\b(?:cat|connectedandroidtest)\b']),

    # Managed devices / aggregators
    ("Gradle", "deviceCheck",                          [rf'(?mi){GRADLE_PREFIX}\s+(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest",            [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),

    # Generic *AndroidTest variant/device tasks (exclude non-test verbs)
    ("Gradle", "variant/device AndroidTest",           [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle", "plain androidTest",                    [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?::[\w-]+:)*androidtest\b']),

    # Third-party runners & CLIs (Gradle tasks)
    ("Gradle", "Spoon",                                [rf'(?mi){GRADLE_PREFIX}[^\n]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon",                             [rf'(?mi){GRADLE_PREFIX}[^\n]*\bmarathon(?:\w*androidtest)?\b']),

    # Other venues (non-gradle)
    ("ADB", "am instrument",                           [r'(?mi)^[^\n]*\bam\s+instrument\b']),
    ("Third_Party_Lab", "gcloud firebase",             [r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "flank",                       [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",                    [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run",               [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),
    ("Flutter", "flutter drive",                       [r'(?mi)^[^\n]*\bflutter\s+drive\b']),
    ("Flutter", "flutter test (integration_test)",     [r'(?mi)^[^\n]*\bflutter\s+test\b[^\n]*\bintegration_test\b']),
    ("Flutter", "dart test (integration_test)",        [r'(?mi)^[^\n]*\bdart\s+test\b[^\n]*\bintegration_test\b']),
]

# === TRIGGER SOURCES (ANYWHERE / lower-confidence; used ONLY in fallback) ===
TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)",                 [rf'(?mi){GRADLE_ANYWHERE}\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle", "connectedAndroidTest (abbr, anywhere)",[rf'(?mi){GRADLE_ANYWHERE}\b(?:cat|connectedandroidtest)\b']),
    ("Gradle", "deviceCheck (anywhere)",               [rf'(?mi){GRADLE_ANYWHERE}\b(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("Gradle", "variant/device AndroidTest (anywhere)",[rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle", "Spoon (anywhere)",                     [rf'(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon (anywhere)",                  [rf'(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b']),
]

# gradle/gradle-build-action inputs (expanded)
GHA_GRADLE_INPUTS = compile_any([
    r'(?mi)^\s*arguments\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*arguments\s*:\s*[\w:-]*androidtest\b',
    r'(?mi)^\s*arguments\s*:\s*\bcat\b',

    r'(?mi)^\s*tasks?\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*tasks?\s*:\s*[\w:-]*androidtest\b',
    r'(?mi)^\s*tasks?\s*:\s*\bcat\b',
])

# Detect "variable-hinted" gradle lines
VAR_HINT = re.compile(r'\${{\s*(?:matrix|env|vars|inputs)\.([^}]+)\s*}}', re.I)
def variable_hints_connected(line: str) -> bool:
    m = VAR_HINT.search(line)
    if not m:
        return False
    hint = m.group(1).lower()
    return any(k in hint for k in [
        "connected","androidtest","device","managed","e2e","ui","espresso","instrument"
    ])

# Precompile groups
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]
TRIGGER_PATTERNS_PRIMARY = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl); groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# === WEAK-HINT GATING & RECONCILIATION ===
STRONG_DEVICE_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner",
    "avdmanager", "sdkmanager system-images/emulator","android create avd",
    "managedDevices DSL", "ManagedVirtualDevice DSL", "GMD task mentions", "GHA gradle arguments/tasks",
    "adb get-state", "adb get-serialno", "adb -s <serial> (physical)",
    "gcloud firebase", "saucectl", "browserstack/bstack", "appcenter test", "maestro cloud",
    "test_matrix/firebase.json",
}
WEAK_DEVICE_LABELS = {"api-level", "abi/arch", "target image", "device name", "sys-img component"}

def filter_weak_device_hints(labels, groups):
    if not (set(labels) & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l not in WEAK_DEVICE_LABELS]
        if not labels:
            groups = []
    return labels, groups

EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator","adb -s emulator-serial"
}
REAL_DEVICE_STRONG_LABELS = {"adb get-state", "adb get-serialno", "adb -s <serial> (physical)"}
REAL_DEVICE_GENERIC_ADB = {"adb devices","adb install", "adb shell", "adb root", "adb settings", "adb input", "adb pm grant"}

def reconcile_emulator_vs_real(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG_LABELS:
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups

def hard_emulator_priority(device_labels, device_groups):
    if ("Emulator" in device_groups
        and "Real_Device" in device_groups
        and not (set(device_labels) & REAL_DEVICE_STRONG_LABELS)):
        device_groups = [g for g in device_groups if g != "Real_Device"]
        device_labels = [l for l in device_labels if l not in REAL_DEVICE_GENERIC_ADB]
    return device_labels, device_groups

# --- Audit helpers ---
SUSPECT_LINE_PAT = re.compile(
    r'(?mi)^\s*[^\n]*\b(?:'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?|'
    r'am\s+instrument|'
    r'gcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run|'
    r'flank\s+android\s+run|'
    r'marathon\b|spoon\b'
    r')[^\n]*$'
)

def audit_reasons(text: str) -> tuple[list[str], list[str]]:
    reasons, lines = [], []
    for m in SUSPECT_LINE_PAT.finditer(text):
        ln = m.group(0).strip()
        if ln:
            lines.append(ln)
        if len(lines) >= 4:
            break

    if re.search(r'(?mi)^\s*(script|run|command)\s*:\s*\|', text) and lines:
        reasons.append("gradle_in_run_block")
    if re.search(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\|)\S', text) and lines:
        reasons.append("gradle_in_run_same_line")
    if re.search(r'(?mi)\bcd\s+\S+\s+&&\s+(?:\./|\.\\)?gradle', text) and lines:
        reasons.append("cd_and_chain")
    if re.search(r'(?mi)^\s*(?:\S+=\S+\s+)+(?:\./|\.\\)?gradle', text) and lines:
        reasons.append("env_prefix_or_abbr_cat")

    if not reasons and lines:
        reasons.append("unknown_trigger_shape")
    return reasons, lines

# === scan & export ===
rows: List[Dict[str, Any]] = []
miss_rows: List[Dict[str, Any]] = []

for fname in os.listdir(CONFIG_DIR):
    ext = os.path.splitext(fname)[1].lower()
    if ext not in ('.yml', '.yaml'):
        continue

    fpath = os.path.join(CONFIG_DIR, fname)
    if not os.path.isfile(fpath):
        continue

    with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()

    # ---------- Primary pass ----------
    content = strip_comments(raw)
    content = re.sub(r'(?m)^\s*-\s*', '', content)
    content = normalize_block_keys(content)

    device_labels, device_groups = collect_hits_with_groups(DEVICE_PATTERNS, content.lower())
    trigger_labels, trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, content.lower())

    trigger_via_prefix = bool(trigger_labels)  # high-confidence

    # gradle/gradle-build-action inputs -> treat as trigger (prefix-confidence)
    if any_match(GHA_GRADLE_INPUTS, content):
        trigger_labels = unique_preserve(trigger_labels + ["gha gradle arguments"])
        trigger_groups = unique_preserve(trigger_groups + ["Gradle"])
        trigger_via_prefix = True

    # Variable-hinted gradle
    if not trigger_via_prefix:
        for line in content.splitlines():
            if re.search(r'(?i)\bgradle(?:w)?(?:\.bat)?\b', line) and variable_hints_connected(line):
                trigger_labels = unique_preserve(trigger_labels + ["gradle via variable hint"])
                trigger_groups = unique_preserve(trigger_groups + ["Gradle"])
                trigger_via_prefix = True
                break

    # WEAK-HINT GATING + RECONCILIATION
    device_labels, device_groups = filter_weak_device_hints(device_labels, device_groups)
    device_labels, device_groups = reconcile_emulator_vs_real(device_labels, device_groups)
    device_labels, device_groups = hard_emulator_priority(device_labels, device_groups)

    # ---------- Fallback ----------
    # Run fallback if NO prefix-based triggers were found (regardless of device presence)
    trigger_via_anywhere = False
    if not trigger_via_prefix:
        fallback = strip_comments(raw)
        fallback = re.sub(r'(?m)^\s*-\s*', '', fallback)
        fallback = re.sub(r'(?mi)^\s*(?:command|run|script)\s*:\s*\|?\s*', '', fallback)
        fallback = re.sub(r'(?m)^\s*sudo\s+', '', fallback)

        # Optionally re-collect device signals if missing (keeps behavior similar to v2.0)
        if not device_labels:
            fb_device_labels, fb_device_groups = collect_hits_with_groups(DEVICE_PATTERNS, fallback.lower())
            fb_device_labels, fb_device_groups = filter_weak_device_hints(fb_device_labels, fb_device_groups)
            fb_device_labels, fb_device_groups = reconcile_emulator_vs_real(fb_device_labels, fb_device_groups)
            if fb_device_labels or fb_device_groups:
                device_labels  = unique_preserve(device_labels  + fb_device_labels)
                device_groups  = unique_preserve(device_groups  + fb_device_groups)
                device_labels, device_groups = hard_emulator_priority(device_labels, device_groups)

        # Now try lower-confidence "anywhere" gradle triggers
        fb_trigger_labels, fb_trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, fallback.lower())
        if any_match(GHA_GRADLE_INPUTS, fallback):
            fb_trigger_labels.append("gha gradle arguments")
            fb_trigger_groups.append("Gradle")

        if fb_trigger_labels:
            trigger_labels = unique_preserve(trigger_labels + fb_trigger_labels)
            trigger_groups = unique_preserve(trigger_groups + fb_trigger_groups)
            trigger_via_anywhere = True

    # parse full_name and ci_platform from: <full_name>__<platform>++<file>.yml
    full_name = "Unknown"; ci_platform = "Unknown"
    base = os.path.basename(fname)
    if "__" in base and "++" in base:
        try:
            full_name = base.split("__", 1)[0]
            ci_platform = base.split("__", 1)[1].split("++", 1)[0]
        except Exception:
            pass

    has_device_setup = bool(device_labels)
    has_test_trigger = bool(trigger_labels)

    real_device_groups = {"Emulator", "GMD", "Third_Party_Lab", "Real_Device"}
    has_real_device_group = any(g in real_device_groups for g in device_groups)
    instru_t_ci = bool(has_test_trigger or (has_device_setup and has_real_device_group))

    miss_reasons_list, miss_lines = [], []
    if has_device_setup and not has_test_trigger:
        miss_reasons_list, miss_lines = audit_reasons(raw)

    row = {
        "filename": fname,
        "full_name": full_name,
        "ci_platform": ci_platform,
        "has_device_setup": has_device_setup,
        "device_setup": ", ".join(device_labels),
        "device_setup_group": ", ".join(device_groups),
        "has_test_trigger": has_test_trigger,
        "test_trigger": ", ".join(trigger_labels),
        "test_trigger_group": ", ".join(trigger_groups),
        "instru_t_ci": instru_t_ci,
        "trigger_via_prefix": bool(trigger_via_prefix),
        "trigger_via_anywhere": bool(trigger_via_anywhere),
        "fall_back": bool(not trigger_via_prefix and (trigger_via_anywhere or has_device_setup)),
        "miss_reasons": ";".join(miss_reasons_list),
        "miss_evidence": " || ".join(miss_lines[:3]),
    }
    rows.append(row)
    if has_device_setup and not has_test_trigger:
        miss_rows.append(row)

# Export main CSV
df = pd.DataFrame(rows, columns=[
    "filename", "full_name", "ci_platform",
    "has_device_setup", "device_setup", "device_setup_group",
    "has_test_trigger", "test_trigger", "test_trigger_group",
    "instru_t_ci",
    "trigger_via_prefix", "trigger_via_anywhere",
     "fall_back",
    "miss_reasons", "miss_evidence",
])
df.to_csv(OUTPUT_CSV, index=False)

# Export misses audit CSV and summary
if miss_rows:
    pd.DataFrame(miss_rows).to_csv(MISSES_CSV, index=False)
    reason_counter: Dict[str, int] = {}
    for r in miss_rows:
        if r["miss_reasons"]:
            for code in r["miss_reasons"].split(";"):
                reason_counter[code] = reason_counter.get(code, 0) + 1
        else:
            reason_counter["(none)"] = reason_counter.get("(none)", 0) + 1
    pd.DataFrame(
        sorted(reason_counter.items(), key=lambda kv: (-kv[1], kv[0])),
        columns=["reason_code", "count"]
    ).to_csv(MISSES_SUMMARY_CSV, index=False)

print(f"✅ Saved main CSV: {OUTPUT_CSV} (yaml files={len(df)})")
if miss_rows:
    print(f"⚠️  Misses audit CSV: {MISSES_CSV} (missed={len(miss_rows)})")
    print(f"📊 Miss reason summary: {MISSES_SUMMARY_CSV}")
else:
    print("🎉 No emulator-without-trigger misses detected.")


✅ Saved main CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1.1_YML_Files_V1.0.csv (yaml files=12667)
⚠️  Misses audit CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_YML_Missed_Triggers_Audit.csv (missed=224)
📊 Miss reason summary: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1_YML_Missed_Triggers_Summary.csv


In [2]:
#Instru Test Signal Config

import os
import re
import pandas as pd

# === CONFIG ===
ROOT = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1.1_Instru_T_Signal_Config.csv"

# --- Helpers ---
def is_build_gradle_file(fname: str) -> bool:
    """
    Accept ONLY Gradle build files, including numbered variants saved with '++':
      - build.gradle / build.gradle.kts
      - build__<n>.gradle / build__<n>.gradle.kts
      - owner.repo__Type++build__<n>.gradle[.kts]
    """
    base = os.path.basename(fname)
    if "++" in base:
        tail = base.split("++", 1)[1]
        tail = re.sub(r"__\d+(?=\.gradle(?:\.kts)?$)", "", tail, flags=re.IGNORECASE).lower()
        return tail in ("build.gradle", "build.gradle.kts")
    return bool(re.match(r"(?i)^build(?:__\d+)?\.gradle(?:\.kts)?$", base))

def extract_full_name(fname: str, fpath: str) -> str:
    """Extract owner.repo from 'owner.repo__Type++...' filenames; fallback to parent dir."""
    base = os.path.basename(fname)
    if "__" in base:
        return base.split("__", 1)[0].lower()
    return os.path.basename(os.path.dirname(fpath)).lower()

def strip_comments_gradle(text: str) -> str:
    """Remove /* ... */ and // ... comments; keep http(s)://."""
    if not text:
        return ""
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.DOTALL)
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.MULTILINE)
    return s

def has_instru_signal_config(text: str) -> bool:
    """
    Instrumentation-test *config* signals in build files (not CI triggers).
    True if any strong signal is present:
      - androidTest dependencies
      - testInstrumentationRunner / args
      - GMD blocks (managedDevices / managedVirtualDevice / deviceGroups + apiLevel)
      - test-only module plugin: com.android.test
      - explicit androidComponents gating/enabling of androidTest
    """
    if not text:
        return False
    t = text.lower()

    # Strong: androidTest deps
    if ("androidtestimplementation" in t
        or "androidtestapi" in t
        or "androidtestcompileonly" in t
        or "androidtestruntimeonly" in t
        or "androidtestcompile" in t):
        return True

    # Runner / runner args
    if "testinstrumentationrunner" in t or "testinstrumentationrunnerarguments" in t:
        return True

    # GMD / managed devices (prefer pair: managed* + apiLevel)
    has_managed = ("manageddevices" in t) or ("managedvirtualdevice" in t) or ("devicegroups" in t)
    if has_managed and "apilevel" in t:
        return True

    # Test-only module plugin
    if 'id("com.android.test")' in t or "id 'com.android.test'" in t or 'apply plugin: "com.android.test"' in t:
        return True

    # Android Components gating (indicates androidTest consideration)
    if "enableandroidtest" in t:
        return True

    return False

# --- Scan & analyze ---
rows = []
for dirpath, _, files in os.walk(ROOT):
    for fname in files:
        if not is_build_gradle_file(fname):
            continue

        fpath = os.path.join(dirpath, fname)
        try:
            with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                raw = f.read()
        except Exception:
            raw = ""

        content = strip_comments_gradle(raw)
        signal = has_instru_signal_config(content)

        rows.append({
            "filename": os.path.basename(fpath),
            "full_name": extract_full_name(fname, fpath),
            "instru_t_signal_config": bool(signal),
        })

# --- Save ---
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV} (rows={len(rows)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\3.1.1_Instru_T_Signal_Config.csv (rows=21280)
